# S3 data usage exploration

Explores the `weekly_report` Storage Lens table (one row per S3 prefix, ~2.8M rows) to see which data types use the most storage.

**All the analysis logic lives in `s3_data_use.py`** so it can be reused elsewhere (e.g. a Flask portal). This notebook only imports those functions and calls them; edit the module, not the notebook, to change behavior.

## Running this notebook

Pick the project `.venv` as the kernel (Select Kernel -> Jupyter Kernel -> "S3 data use exploration (.venv)"). Credentials are only needed to *refresh* the cached data; the cells below read the cached `weekly_report.parquet` and `capsule_map.json`, so they run offline.

To refresh from source: run `aws sso login --profile aind_prod_sso`, then `s3.load_weekly_report(use_cache=False)` (re-queries the RDS table), and `resolve_capsules.py` (rebuilds `capsule_map.json`, needs the Code Ocean token).

In [18]:
import pandas as pd

import s3_data_use as s3

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.options.display.float_format = "{:,.2f}".format

# Load the cached snapshot and attach the data-type `category` column.
df = s3.load_weekly_report()      # reads weekly_report.parquet (use_cache=False re-queries RDS)
df = s3.add_category(df)          # reads capsule_map.json to resolve Code Ocean pipelines
print(f"{len(df):,} prefixes, {df['size_tb'].sum():,.0f} TB, {df['category'].nunique()} categories")

2,817,972 prefixes, 30,104 TB, 227 categories


## Size by bucket

In [19]:
s3.size_by_bucket(df)

,total_tb
bucket,
aind-open-data,"20,304.71"
codeocean-s3resultsbucket-1182nktl2bh9f,"3,497.47"
aind-archive,"1,846.75"
codeocean-s3datasetsbucket-1u41qdg42ur9,"1,772.62"
aind-scratch-data,"1,365.67"
codeocean-s3batchbucket-16itpvq060udk,575.73
aind-private-data-prod-o5171v,549.61
aind-msma-morphology-data,91.02
aind-ephys-data,49.96


## Size by category

A category is a data-type/modality label assigned to every prefix. See the `s3_data_use` module docstring for how each prefix is labeled (named-prefix token, resolved Code Ocean pipeline, or bucket fallback), and `s3.MODALITY_RULES` for the keyword map you can edit.

In [20]:
s3.size_by_category(df)

,total_tb,n_prefixes
category,,
exaSPIM,"10,247.88",973
SmartSPIM,"7,292.19",9208
multiplane-ophys,"1,571.09",3941
codeocean-pipeline (private/unresolved),"1,391.67",959
HCR,"1,296.27",3143
codeocean-dataset (unresolved),"1,045.44",48569
timwang,"1,024.58",1
behavior,703.36,35777
rachel.dalley,644.79,1


## Category by bucket (main view)

TB by category (rows) x bucket (columns). Bucket columns are ordered by total size (largest, `aind-open-data`, on the left); category rows are sorted by `aind-open-data` usage. The bottom `TOTAL` row is the true per-bucket size and the `(other categories)` row rolls up everything below the top 20, so every column adds up. Pass `sort_bucket=`, `top_n=`, or `round_tb=` to customize.

In [21]:
s3.category_by_bucket(df, sort_bucket="aind-open-data")

bucket,aind-open-data,codeocean-s3resultsbucket-1182nktl2bh9f,aind-archive,codeocean-s3datasetsbucket-1u41qdg42ur9,aind-scratch-data,codeocean-s3batchbucket-16itpvq060udk,aind-private-data-prod-o5171v,aind-msma-morphology-data,aind-ephys-data,aind-ophys-data,aind-behavior-data,aind-private-scratch-data-prod-o5171v,aind-open-data-prod-o5171v,TOTAL
category,,,,,,,,,,,,,,
exaSPIM,"10,153.74",24.29,0.00,0.00,68.24,1.56,0.00,0.06,0.00,0.00,0.00,0.00,0.00,"10,247.88"
SmartSPIM,"6,690.47",54.03,0.00,0.00,129.08,416.05,0.58,0.00,0.00,0.00,0.00,0.00,1.98,"7,292.19"
HCR,"1,271.97",21.15,0.00,0.00,2.65,0.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"1,296.27"
aind-open-data (uuid),483.45,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,483.45
multiplane-ophys,463.88,"1,005.76",0.00,0.00,0.02,1.03,100.40,0.00,0.00,0.00,0.00,0.00,0.00,"1,571.09"
behavior,405.81,58.89,3.30,0.00,38.88,0.03,195.43,0.00,0.00,0.00,1.03,0.00,0.00,703.36
fMOST,280.14,0.00,0.00,0.00,0.00,0.00,0.00,20.46,0.00,0.00,0.00,0.00,0.00,300.60
ecephys,215.81,77.68,0.06,0.00,0.76,85.56,167.10,0.00,49.95,0.00,0.00,0.00,0.00,596.92
single-plane-ophys,49.00,57.84,0.00,0.00,1.58,0.00,70.77,0.00,0.00,0.19,0.00,0.00,0.00,179.38


## One call for an app (e.g. Dan's Flask portal)

`build_category_by_bucket()` does load -> add category -> pivot in one shot from the cached artifacts (no AWS/Code Ocean credentials needed) and returns a DataFrame. A Flask view can render it directly with `.to_html(...)`.

In [22]:
table = s3.build_category_by_bucket(sort_bucket="aind-open-data")

# In a Flask view this is what you'd hand to the template:
html = table.to_html(classes="table table-sm", border=0, float_format=lambda x: f"{x:,.1f}")
print(html[:400], "...")
table

<table class="dataframe table table-sm">
  <thead>
    <tr style="text-align: right;">
      <th>bucket</th>
      <th>aind-open-data</th>
      <th>codeocean-s3resultsbucket-1182nktl2bh9f</th>
      <th>aind-archive</th>
      <th>codeocean-s3datasetsbucket-1u41qdg42ur9</th>
      <th>aind-scratch-data</th>
      <th>codeocean-s3batchbucket-16itpvq060udk</th>
      <th>aind-private-data-prod-o517 ...


bucket,aind-open-data,codeocean-s3resultsbucket-1182nktl2bh9f,aind-archive,codeocean-s3datasetsbucket-1u41qdg42ur9,aind-scratch-data,codeocean-s3batchbucket-16itpvq060udk,aind-private-data-prod-o5171v,aind-msma-morphology-data,aind-ephys-data,aind-ophys-data,aind-behavior-data,aind-private-scratch-data-prod-o5171v,aind-open-data-prod-o5171v,TOTAL
category,,,,,,,,,,,,,,
exaSPIM,"10,153.74",24.29,0.00,0.00,68.24,1.56,0.00,0.06,0.00,0.00,0.00,0.00,0.00,"10,247.88"
SmartSPIM,"6,690.47",54.03,0.00,0.00,129.08,416.05,0.58,0.00,0.00,0.00,0.00,0.00,1.98,"7,292.19"
HCR,"1,271.97",21.15,0.00,0.00,2.65,0.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"1,296.27"
aind-open-data (uuid),483.45,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,483.45
multiplane-ophys,463.88,"1,005.76",0.00,0.00,0.02,1.03,100.40,0.00,0.00,0.00,0.00,0.00,0.00,"1,571.09"
behavior,405.81,58.89,3.30,0.00,38.88,0.03,195.43,0.00,0.00,0.00,1.03,0.00,0.00,703.36
fMOST,280.14,0.00,0.00,0.00,0.00,0.00,0.00,20.46,0.00,0.00,0.00,0.00,0.00,300.60
ecephys,215.81,77.68,0.06,0.00,0.76,85.56,167.10,0.00,49.95,0.00,0.00,0.00,0.00,596.92
single-plane-ophys,49.00,57.84,0.00,0.00,1.58,0.00,70.77,0.00,0.00,0.19,0.00,0.00,0.00,179.38
